**Nama** Adelgun Roman Seran

**Nim** 2301020087

**Nama Proyek:** Analisis sentimen teks Bahasa Indonesia  
**Tema:** Analisis Sentimen  
**Tools:** Python, pandas, scikit-learn, Sastrawi, matplotlib, wordcloud  

### Tujuan
Notebook ini membangun sistem NLP sederhana untuk:
1. melakukan **text preprocessing**,
2. merepresentasikan teks dengan **Bag-of-Words** dan **TF-IDF**,
3. membandingkan hasil kedua metode pada tugas **klasifikasi sentimen**,
4. menampilkan visualisasi yang membantu analisis hasil.

### Dataset
Notebook ini menggunakan dataset sentimen publik Bahasa Indonesia dari repositori multilingual sentiment datasets.  
File yang dipakai adalah **`indonesian/test.csv`** yang berisi **500 teks**.

Sumber dataset:
- `https://raw.githubusercontent.com/tyqiangz/multilingual-sentiment-datasets/main/data/indonesian/test.csv`
- `https://github.com/tyqiangz/multilingual-sentiment-datasets`

### Catatan
- Preprocessing yang digunakan: **cleaning, tokenization, stopword removal, dan stemming**.
- Metode representasi teks yang dibandingkan: **Bag-of-Words** dan **TF-IDF**.
- Model klasifikasi yang digunakan sama untuk kedua representasi agar perbandingan tetap adil, yaitu **Logistic Regression**.


In [ ]:
!pip -q install Sastrawi wordcloud

In [ ]:

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from functools import lru_cache
from wordcloud import WordCloud

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_colwidth", 120)



## 1. Memuat Dataset

Cell berikut mencoba membaca dataset dari file lokal terlebih dahulu.  
Jika file lokal tidak ada, notebook akan mengambil langsung dari sumber publik GitHub.


In [ ]:

LOCAL_PATH = '/content/indonesian_test.csv'
URL = 'https://raw.githubusercontent.com/tyqiangz/multilingual-sentiment-datasets/main/data/indonesian/test.csv'

try:
    df = pd.read_csv(LOCAL_PATH)
    source_used = LOCAL_PATH
except:
    df = pd.read_csv(URL)
    source_used = URL

print('Sumber data:', source_used)
print('Ukuran data:', df.shape)
df.head()


In [ ]:

df['label'].value_counts()



## 2. Analisis Awal Data
Visualisasi pertama menampilkan distribusi label sentimen.


In [ ]:

label_counts = df['label'].value_counts().sort_index()

plt.figure(figsize=(7,4))
label_counts.plot(kind='bar')
plt.title('Distribusi Kelas Sentimen')
plt.xlabel('Label')
plt.ylabel('Jumlah Dokumen')
plt.tight_layout()
plt.show()



## 3. Text Preprocessing

Tahap preprocessing yang dilakukan:
1. **Cleaning**: mengubah huruf menjadi lowercase, menghapus URL, mention, hashtag, angka, dan simbol.
2. **Tokenization**: memecah teks menjadi token berdasarkan spasi.
3. **Stopword Removal**: menghapus kata umum yang kurang informatif.
4. **Stemming**: mengubah kata ke bentuk dasar menggunakan **Sastrawi**.

Bagian ini penting karena kualitas representasi teks sangat dipengaruhi oleh kebersihan teks.


In [ ]:

stemmer = StemmerFactory().create_stemmer()
stopwords = set(StopWordRemoverFactory().get_stop_words())

extra_stopwords = {
    'yg','dg','rt','dgn','ny','d','klo','kalo','amp','biar','bikin','bilang',
    'gak','ga','nggak','ngga','gue','gua','aku','sih','nih','nya','aja',
    'deh','dong','lah','kok','ya','jadi','udah','sudah'
}
stopwords |= extra_stopwords

@lru_cache(maxsize=50000)
def stem_word(word):
    return stemmer.stem(word)

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'@\w+|#\w+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    tokens = text.split()
    tokens = [t for t in tokens if len(t) > 2 and t not in stopwords]
    stems = [stem_word(t) for t in tokens]

    return ' '.join(stems)


In [ ]:

df = df.dropna(subset=['text', 'label']).copy()
df['clean_text'] = df['text'].apply(preprocess_text)
df = df[df['clean_text'].str.strip() != ''].copy()

print('Ukuran data setelah preprocessing:', df.shape)
df[['text', 'clean_text', 'label']].head()



## 4. Visualisasi Teks

Visualisasi kedua dan ketiga:
- **Word Cloud**
- **20 token paling sering muncul**


In [ ]:

all_text = ' '.join(df['clean_text'])
wordcloud = WordCloud(width=1200, height=600, background_color='white').generate(all_text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud Setelah Preprocessing')
plt.show()


In [ ]:

token_counts = Counter(all_text.split())
top20 = pd.DataFrame(token_counts.most_common(20), columns=['token', 'frekuensi']).iloc[::-1]

plt.figure(figsize=(8,6))
plt.barh(top20['token'], top20['frekuensi'])
plt.title('20 Token Paling Sering Muncul')
plt.xlabel('Frekuensi')
plt.tight_layout()
plt.show()

top20.iloc[::-1]



## 5. Menyiapkan Data Latih dan Data Uji
Pembagian data menggunakan **80:20** dengan stratifikasi agar proporsi label tetap seimbang.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print('Jumlah data latih:', len(X_train))
print('Jumlah data uji  :', len(X_test))



## 6. Eksperimen 1: Bag-of-Words
Metode pertama menggunakan **CountVectorizer** sebagai representasi Bag-of-Words.


In [ ]:

bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1,2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

bow_model = LogisticRegression(max_iter=1000)
bow_model.fit(X_train_bow, y_train)

y_pred_bow = bow_model.predict(X_test_bow)

bow_accuracy = accuracy_score(y_test, y_pred_bow)
bow_f1 = f1_score(y_test, y_pred_bow, average='macro')

print('Accuracy Bag-of-Words :', round(bow_accuracy, 4))
print('Macro F1 Bag-of-Words :', round(bow_f1, 4))
print()
print(classification_report(y_test, y_pred_bow))


In [ ]:

labels_order = ['negative', 'neutral', 'positive']
cm_bow = confusion_matrix(y_test, y_pred_bow, labels=labels_order)

plt.figure(figsize=(5,4))
plt.imshow(cm_bow, cmap='Blues')
plt.title('Confusion Matrix - Bag-of-Words')
plt.xticks(range(len(labels_order)), labels_order)
plt.yticks(range(len(labels_order)), labels_order)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')

for i in range(cm_bow.shape[0]):
    for j in range(cm_bow.shape[1]):
        plt.text(j, i, cm_bow[i, j], ha='center', va='center')

plt.tight_layout()
plt.show()



## 7. Eksperimen 2: TF-IDF
Metode kedua menggunakan **TfidfVectorizer**.


In [ ]:

tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_train_tfidf, y_train)

y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

tfidf_accuracy = accuracy_score(y_test, y_pred_tfidf)
tfidf_f1 = f1_score(y_test, y_pred_tfidf, average='macro')

print('Accuracy TF-IDF :', round(tfidf_accuracy, 4))
print('Macro F1 TF-IDF :', round(tfidf_f1, 4))
print()
print(classification_report(y_test, y_pred_tfidf))


In [ ]:

cm_tfidf = confusion_matrix(y_test, y_pred_tfidf, labels=labels_order)

plt.figure(figsize=(5,4))
plt.imshow(cm_tfidf, cmap='Greens')
plt.title('Confusion Matrix - TF-IDF')
plt.xticks(range(len(labels_order)), labels_order)
plt.yticks(range(len(labels_order)), labels_order)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')

for i in range(cm_tfidf.shape[0]):
    for j in range(cm_tfidf.shape[1]):
        plt.text(j, i, cm_tfidf[i, j], ha='center', va='center')

plt.tight_layout()
plt.show()



## 8. Perbandingan Hasil
Karena model yang dipakai sama, perbedaan hasil terutama dipengaruhi oleh representasi teks.


In [ ]:

comparison = pd.DataFrame({
    'Metode': ['Bag-of-Words', 'TF-IDF'],
    'Accuracy': [bow_accuracy, tfidf_accuracy],
    'Macro F1': [bow_f1, tfidf_f1]
}).sort_values(by='Accuracy', ascending=False)

comparison



### Interpretasi Singkat
- Pada eksperimen ini, **Bag-of-Words** memberikan nilai **accuracy** dan **macro F1** sedikit lebih baik daripada **TF-IDF**.
- Salah satu penyebabnya adalah ukuran dataset yang relatif kecil, sehingga representasi frekuensi sederhana masih cukup efektif.
- Kelas **neutral** menjadi kelas yang paling sulit diprediksi karena sering memiliki kata yang tumpang tindih dengan kelas lain.

## 9. Uji Prediksi Kalimat Baru
Cell berikut dapat digunakan untuk menguji kalimat baru.


In [ ]:

def predict_sentiment(text, method='bow'):
    cleaned = preprocess_text(text)

    if method.lower() == 'bow':
        vec = bow_vectorizer.transform([cleaned])
        pred = bow_model.predict(vec)[0]
    else:
        vec = tfidf_vectorizer.transform([cleaned])
        pred = tfidf_model.predict(vec)[0]

    return {'text_asli': text, 'text_clean': cleaned, 'prediksi': pred}

contoh_1 = predict_sentiment('Aplikasinya sangat membantu dan fiturnya bagus sekali', method='bow')
contoh_2 = predict_sentiment('Pelayanannya buruk dan saya sangat kecewa', method='tfidf')

contoh_1, contoh_2



## 10. Kesimpulan

1. menggunakan **Python** dan library NLP yang relevan,
2. melakukan preprocessing lengkap,
3. menerapkan **dua metode representasi teks**,
4. membandingkan performa kedua metode,
5. menggunakan dataset **500 teks**,
6. menyajikan lebih dari dua visualisasi,
7. dapat dijalankan ulang di Google Colab.

### Ringkasan Akhir
- Dataset: **500 teks Bahasa Indonesia**
- Preprocessing: **cleaning, tokenization, stopword removal, stemming**
- Representasi: **Bag-of-Words** dan **TF-IDF**
- Model: **Logistic Regression**
- Hasil terbaik pada notebook ini diperoleh oleh **Bag-of-Words**
